# MuJoCo + reinforcement learning on Kaggle
Settings (right panel): **Internet = On** (needed for `pip`). GPU not needed.
Run the cells in order. The project files are written by the `%%writefile` cells.

In [ ]:
!pip install -q mujoco gymnasium stable-baselines3
import mujoco; print('mujoco', mujoco.__version__)

In [ ]:
!mkdir -p robots src/mujoco_sim/envs

In [ ]:
%%writefile robots/arm.urdf
<?xml version="1.0"?>
<robot name="two_link_arm">
  <link name="base">
    <inertial><origin xyz="0 0 0.05"/><mass value="5"/><inertia ixx="0.1" iyy="0.1" izz="0.1" ixy="0" ixz="0" iyz="0"/></inertial>
    <visual><geometry><cylinder radius="0.08" length="0.1"/></geometry><origin xyz="0 0 0.05"/></visual>
  </link>
  <link name="upper">
    <inertial><origin xyz="0 0 0.15"/><mass value="1"/><inertia ixx="0.01" iyy="0.01" izz="0.001" ixy="0" ixz="0" iyz="0"/></inertial>
    <visual><geometry><box size="0.05 0.05 0.3"/></geometry><origin xyz="0 0 0.15"/></visual>
    <collision><geometry><box size="0.05 0.05 0.3"/></geometry><origin xyz="0 0 0.15"/></collision>
  </link>
  <link name="fore">
    <inertial><origin xyz="0 0 0.125"/><mass value="0.5"/><inertia ixx="0.005" iyy="0.005" izz="0.0005" ixy="0" ixz="0" iyz="0"/></inertial>
    <visual><geometry><box size="0.04 0.04 0.25"/></geometry><origin xyz="0 0 0.125"/></visual>
    <collision><geometry><box size="0.04 0.04 0.25"/></geometry><origin xyz="0 0 0.125"/></collision>
  </link>
  <joint name="shoulder" type="revolute"><parent link="base"/><child link="upper"/><origin xyz="0 0 0.1"/><axis xyz="0 1 0"/><limit lower="-1.5" upper="1.5" effort="20" velocity="2"/></joint>
  <joint name="elbow" type="revolute"><parent link="upper"/><child link="fore"/><origin xyz="0 0 0.3"/><axis xyz="0 1 0"/><limit lower="-2" upper="2" effort="10" velocity="2"/></joint>
</robot>


In [ ]:
%%writefile src/mujoco_sim/scene.py
"""Build a MuJoCo scene from a URDF robot file and simulate it."""

import mujoco
import numpy as np


def build_model(
    urdf_path: str,
    control: str = "torque",
    tip_body: str | None = None,
    tip_offset: tuple = (0.0, 0.0, 0.0),
    floor: bool = True,
    kp: float = 50.0,
    kv: float = 8.0,
    max_force: float = 20.0,
) -> mujoco.MjModel:
    """Load a URDF and complete it with what URDF cannot describe.

    URDF has no floor, light, camera or motor, so they are added here:
    a floor, a light, a camera named "front", and one motor per joint.

    Args:
        urdf_path (str): Path to the URDF file.
        control (str): "torque" for direct torque motors (ctrl in [-5, 5]), or "position"
            for position servos (ctrl = target joint angle within the joint range).
        tip_body (str | None): Name of the body that carries the end effector. If given,
            a site named "tip" is added to it.
        tip_offset (tuple): Position of the "tip" site in the frame of `tip_body`.
        floor (bool): Whether to add a floor.
        kp (float): Stiffness of the position servos.
        kv (float): Damping of the position servos.
        max_force (float): Torque limit of the position servos.

    Returns:
        mujoco.MjModel: Compiled model ready for simulation.
    """
    if control not in ("torque", "position"):
        raise ValueError(f"control must be 'torque' or 'position', got {control!r}")
    spec = mujoco.MjSpec.from_file(str(urdf_path))
    if control == "position":
        # The servo damping is stiff for these light links: with the default explicit
        # Euler integrator the joints chatter and never settle. Implicit integration does.
        spec.option.integrator = mujoco.mjtIntegrator.mjINT_IMPLICITFAST
    if floor:
        spec.worldbody.add_geom(
            name="floor", type=mujoco.mjtGeom.mjGEOM_PLANE, size=[2, 2, 0.1], rgba=[0.8, 0.8, 0.8, 1]
        )
    spec.worldbody.add_light(pos=[0, 0, 2], dir=[0, 0, -1])
    spec.worldbody.add_camera(name="front", pos=[1.2, 0, 0.5], xyaxes=[0, 1, 0, -0.3, 0, 1])
    for joint in spec.joints:
        motor = spec.add_actuator(name=joint.name, target=joint.name, trntype=mujoco.mjtTrn.mjTRN_JOINT)
        if control == "torque":
            motor.ctrllimited = True
            motor.ctrlrange = [-5, 5]
        else:
            motor.gaintype = mujoco.mjtGain.mjGAIN_FIXED
            motor.biastype = mujoco.mjtBias.mjBIAS_AFFINE
            motor.gainprm[0] = kp
            motor.biasprm[1] = -kp
            motor.biasprm[2] = -kv
            motor.ctrllimited = True
            motor.ctrlrange = list(joint.range)
            motor.forcelimited = True
            motor.forcerange = [-max_force, max_force]
    if tip_body is not None:
        spec.body(tip_body).add_site(name="tip", pos=list(tip_offset), size=[0.01, 0, 0])
    return spec.compile()


def rollout(
    model: mujoco.MjModel,
    n_frames: int = 8,
    steps_per_frame: int = 50,
    size: int = 224,
    render: bool = True,
) -> tuple[np.ndarray | None, np.ndarray]:
    """Move the joints with a sine torque and record the state (and optionally images).

    Args:
        model (mujoco.MjModel): Model returned by `build_model` (torque control).
        n_frames (int): Number of recorded frames.
        steps_per_frame (int): Physics steps between two frames.
        size (int): Height and width of the rendered images.
        render (bool): If False, only the physics runs (no graphics needed).

    Returns:
        tuple: (frames, qpos). `frames` has shape (n_frames, size, size, 3), dtype uint8,
            or is None when render is False. `qpos` has shape (n_frames, n_joints).
    """
    data = mujoco.MjData(model)
    renderer = mujoco.Renderer(model, size, size) if render else None
    frames, qpos = [], []
    for i in range(n_frames):
        data.ctrl[:] = 3.0 * np.sin(0.8 * i + np.arange(model.nu))
        for _ in range(steps_per_frame):
            mujoco.mj_step(model, data)
        qpos.append(data.qpos.copy())
        if renderer is not None:
            renderer.update_scene(data, camera="front")
            frames.append(renderer.render().copy())
    return (np.stack(frames) if render else None), np.stack(qpos)


In [ ]:
%%writefile src/mujoco_sim/__init__.py


In [ ]:
%%writefile src/mujoco_sim/envs/__init__.py
from mujoco_sim.envs.reach import ArmReachEnv

__all__ = ["ArmReachEnv"]


In [ ]:
%%writefile src/mujoco_sim/envs/reach.py
"""Gymnasium environment: move the tip of a robot arm to a target position."""

from pathlib import Path

import gymnasium as gym
import mujoco
import numpy as np
from gymnasium import spaces

from mujoco_sim.scene import build_model

DEFAULT_URDF = Path(__file__).resolve().parents[3] / "robots" / "arm.urdf"


class ArmReachEnv(gym.Env):
    """Reaching task for the two-joint arm of `robots/arm.urdf`.

    The arm moves in the x-z plane. A target is sampled in the reachable workspace
    (the tip position of a random joint configuration), and the agent must bring the
    arm tip within `success_radius` of it.

    Action (2,): target joint angles, normalized to [-1, 1] over each joint range
        (position servos).
    Observation (8,): joint positions normalized to [-1, 1], joint velocities,
        target (x, z) and tip-to-target vector (x, z).
    Reward: minus the distance between tip and target, plus 1 on success.
    The episode ends on success (terminated) or after `max_steps` steps (truncated).
    """

    metadata = {"render_modes": []}

    def __init__(
        self,
        urdf_path: str | Path = DEFAULT_URDF,
        frame_skip: int = 10,
        max_steps: int = 100,
        success_radius: float = 0.04,
        min_target_height: float = 0.05,
        min_target_distance: float = 0.15,
    ) -> None:
        """
        Args:
            urdf_path: URDF of a robot with bodies "fore" (end link) and revolute joints.
            frame_skip: Physics steps per environment step (control period = frame_skip * 0.002 s).
            max_steps: Episode length.
            success_radius: Distance (m) under which the target counts as reached.
            min_target_height: Targets are sampled above this height (m).
            min_target_distance: Targets are at least this far (m) from the starting tip position.
        """
        super().__init__()
        self.model = build_model(
            str(urdf_path), control="position", tip_body="fore", tip_offset=(0.0, 0.0, 0.25), floor=False
        )
        self.data = mujoco.MjData(self.model)
        self._fk_data = mujoco.MjData(self.model)
        self.frame_skip = frame_skip
        self.max_steps = max_steps
        self.success_radius = success_radius
        self.min_target_height = min_target_height
        self.min_target_distance = min_target_distance
        self.q_low = self.model.jnt_range[:, 0].copy()
        self.q_high = self.model.jnt_range[:, 1].copy()
        n = self.model.nu
        self.action_space = spaces.Box(-1.0, 1.0, (n,), np.float32)
        self.observation_space = spaces.Box(-np.inf, np.inf, (2 * n + 4,), np.float32)
        self.target = np.zeros(2)
        self._steps = 0

    def tip_position(self, data: mujoco.MjData | None = None) -> np.ndarray:
        """Tip position (x, z) for the given data (default: the current state)."""
        data = self.data if data is None else data
        return data.site_xpos[0][[0, 2]].copy()

    def forward_kinematics(self, q: np.ndarray) -> np.ndarray:
        """Tip position (x, z) for joint angles `q`, without disturbing the simulation."""
        self._fk_data.qpos[:] = q
        mujoco.mj_forward(self.model, self._fk_data)
        return self.tip_position(self._fk_data)

    def _sample_target(self, start_tip: np.ndarray) -> np.ndarray:
        for _ in range(100):
            q = self.np_random.uniform(self.q_low, self.q_high)
            tip = self.forward_kinematics(q)
            if tip[1] >= self.min_target_height and np.linalg.norm(tip - start_tip) >= self.min_target_distance:
                return tip
        return tip

    def _observation(self) -> np.ndarray:
        q = self.data.qpos
        q_norm = 2.0 * (q - self.q_low) / (self.q_high - self.q_low) - 1.0
        tip = self.tip_position()
        return np.concatenate([q_norm, self.data.qvel, self.target, self.target - tip]).astype(np.float32)

    def reset(self, *, seed: int | None = None, options: dict | None = None):
        """Reset the arm near its upright pose and sample a target.

        `options={"target": [x, z]}` forces the target (used for tests and demos).
        """
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[:] = self.np_random.uniform(-0.05, 0.05, self.model.nu)
        self.data.ctrl[:] = self.data.qpos
        mujoco.mj_forward(self.model, self.data)
        if options and "target" in options:
            self.target = np.asarray(options["target"], dtype=np.float64)
        else:
            self.target = self._sample_target(self.tip_position())
        self._steps = 0
        return self._observation(), {"distance": float(np.linalg.norm(self.target - self.tip_position()))}

    def step(self, action):
        action = np.clip(np.asarray(action, dtype=np.float64), -1.0, 1.0)
        self.data.ctrl[:] = self.q_low + (action + 1.0) / 2.0 * (self.q_high - self.q_low)
        for _ in range(self.frame_skip):
            mujoco.mj_step(self.model, self.data)
        self._steps += 1
        distance = float(np.linalg.norm(self.target - self.tip_position()))
        success = distance < self.success_radius
        reward = -distance + (1.0 if success else 0.0)
        truncated = self._steps >= self.max_steps and not success
        return self._observation(), reward, success, truncated, {"distance": distance, "is_success": success}


## 1. MuJoCo runs (physics only)

In [ ]:
import sys; sys.path.insert(0, 'src')
from mujoco_sim.scene import build_model, rollout
model = build_model('robots/arm.urdf')
_, qpos = rollout(model, render=False)
print(qpos.round(2))

## 2. Rendering (optional, needs OpenGL; if it fails, skip it)

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'
try:
    frames, _ = rollout(model, n_frames=4)
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 4, figsize=(12, 3))
    for a, f in zip(ax, frames): a.imshow(f); a.axis('off')
    plt.show()
except Exception as e:
    print('Rendering not available here:', e)

## 3. Reinforcement learning: PPO on the reaching task
Expected (local run): 300k steps -> about 100 % success vs 0 % for zero action and ~10 % for random. Takes a few minutes.

In [ ]:
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from mujoco_sim.envs import ArmReachEnv

def evaluate(policy, episodes=200, seed=1000):
    env = ArmReachEnv(); ok, dist = [], []
    for ep in range(episodes):
        obs, _ = env.reset(seed=seed + ep); done = False
        while not done:
            obs, _, term, trunc, info = env.step(policy(obs)); done = term or trunc
        ok.append(info['is_success']); dist.append(info['distance'])
    return float(np.mean(ok)), float(np.mean(dist))

vec = make_vec_env(ArmReachEnv, n_envs=4, seed=0)
ppo = PPO('MlpPolicy', vec, seed=0, device='cpu')
ppo.learn(total_timesteps=300_000)
rng = np.random.default_rng(0)
for name, pol in [('zero action', lambda o: np.zeros(2)), ('random', lambda o: rng.uniform(-1, 1, 2)), ('PPO', lambda o: ppo.predict(o, deterministic=True)[0])]:
    s, d = evaluate(pol); print(f'{name:<12} success {s:.2f}  final distance {d:.3f} m')